# 第3日 ノートブック03：検出結果を読み解く

# Day 3 Notebook 03: Understanding Detection Results

---

## 学習目標 / Learning objectives

| 日本語 | English |
| --- | --- |
| 1. confidence のしきい値を変えて結果を比較できる | 1. Compare results across confidence thresholds |
| 2. 検出結果の座標を読み、意味を説明できる | 2. Read detection coordinates and explain them |
| 3. 偽陽性と偽陰性を、実際の画像で指摘できる | 3. Point out false positives and false negatives |
| 4. 画像サイズを下げると検出できなくなる理由を説明できる | 4. Explain why shrinking an image loses detections |
| 5. 実験結果を表にまとめて報告できる | 5. Record experiments in a table and report them |

## このノートブックの基本情報 / Notebook facts

| 項目 / Item | 内容 / Details |
| --- | --- |
| 学習時間（目安） / Estimated duration | 120 分 / 120 minutes |
| 難易度 / Difficulty | 初級 / Beginner |
| 前提条件 / Prerequisites | 第3日ノートブック01・02 を終えていること。 / You have finished Day 3 Notebooks 01 and 02. |
| 必要なライブラリ / Required libraries | torch, ultralytics, opencv-python, numpy, matplotlib, pandas |
| 使用カーネル / Jupyter kernel | aivision |

### 提出物 / Expected deliverables

| 日本語 | English |
| --- | --- |
| しきい値ごとの検出数の表 | A table of detection counts per threshold |
| 失敗例の記録（1 件以上） | At least one recorded failure case |

## 本日の学習タイムライン / Learning timeline for today

| 時刻 / Time | 内容（日本語） | Activity (English) |
| --- | --- | --- |
| 08:00–09:00 | 講師：しきい値と失敗例の説明 | Instructor: thresholds and failures |
| 09:00–10:30 | Part 1〜Part 3（しきい値・座標・比較） | Part 1-Part 3 (thresholds, coordinates, comparison) |
| 10:30–10:40 | 休憩 | Short break |
| 10:40–12:00 | Part 4〜Part 5（失敗の理由・解像度） | Part 4-Part 5 (failure causes, resolution) |
| 12:00–12:50 | 昼休み | Lunch break |
| 12:50–13:50 | 講師：振り返りと午後の説明 | Instructor: review and briefing |
| 13:50–16:50 | 演習・提出物 | Exercises and submission |

> **日本語：** 12:00–12:50 は昼休みです。08:00–09:00 と 12:50–13:50 は講師の時間、それ以外は自分のペースで進めてください。
>
> **English:** 12:00-12:50 is the lunch break. 08:00-09:00 and 12:50-13:50 are instructor sessions; the remaining time is self-paced work.

---

## Part 0　準備 / Preparation

In [ ]:
from pathlib import Path

def find_course_root(start=None):
    """研修フォルダ（AI_Vision_Training）を上方向に探す。
    Walk upwards until the course root folder is found."""
    current = Path(start).resolve() if start else Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "assets").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError(
        "研修フォルダが見つかりません。JupyterLab を AI_Vision_Training で起動してください。\n"
        "Course folder not found. Start JupyterLab inside the AI_Vision_Training folder."
    )

COURSE_ROOT = find_course_root()
ASSETS = COURSE_ROOT / "assets"
print("研修フォルダ / course root :", COURSE_ROOT)
print("素材フォルダ / assets      :", ASSETS)

In [ ]:
import importlib
import sys

print("Python version:", sys.version.split()[0])

REQUIRED = [
    ("numpy", "numpy"),
    ("matplotlib", "matplotlib"),
    ("cv2", "opencv-python"),
    ("torch", "torch"),
    ("ultralytics", "ultralytics"),
    ("pandas", "pandas"),
]

missing = []
for import_name, pip_name in REQUIRED:
    try:
        module = importlib.import_module(import_name)
        version = getattr(module, "__version__", "unknown")
        print(f"  [OK]      {pip_name:<20} {version}")
    except ImportError:
        print(f"  [MISSING] {pip_name:<20} -> pip install {pip_name}")
        missing.append(pip_name)

if missing:
    print("\n未導入のパッケージがあります / Some packages are missing:", ", ".join(missing))
    print("docs/troubleshooting/installation.md を確認してください / see that file for help.")
else:
    print("\nこのノートブックに必要なライブラリはすべて揃っています。")
    print("All libraries required by this notebook are available.")

In [ ]:
import platform
from pathlib import Path

from matplotlib import font_manager, rcParams

def find_japanese_font():
    """日本語を表示できるフォントを探す。見つからなければ None を返す。
    Locate a font that can render Japanese. Returns None when nothing is found."""
    system = platform.system()
    if system == "Windows":
        folders = [Path("C:/Windows/Fonts")]
    elif system == "Darwin":
        folders = [Path("/System/Library/Fonts"), Path("/Library/Fonts"), Path.home() / "Library/Fonts"]
    else:
        folders = [Path("/usr/share/fonts"), Path("/usr/local/share/fonts"), Path.home() / ".fonts"]

    preferred = [
        "NotoSansJP-Regular", "NotoSansCJK-Regular", "YuGothR", "meiryo", "msgothic",
        "BIZ-UDGothicR", "NotoSansJP", "HiraginoSans",
    ]
    found = []
    for folder in folders:
        if not folder.is_dir():
            continue
        for path in folder.rglob("*"):
            if path.suffix.lower() not in {".ttf", ".ttc", ".otf"}:
                continue
            name = path.stem
            if any(key.lower() in name.lower() for key in preferred):
                found.append(path)

    def rank(path):
        name = path.stem.lower()
        return (0 if "regular" in name or name.endswith("r") else 1, len(name))

    return sorted(found, key=rank)[0] if found else None


japanese_font = find_japanese_font()
if japanese_font is None:
    print("日本語フォントが見つかりません / no Japanese font found.")
    print("図のタイトルが □ になる場合は、英字のタイトルを使ってください。")
    print("If figure titles show boxes, use English titles instead.")
else:
    font_manager.fontManager.addfont(str(japanese_font))
    family_name = font_manager.FontProperties(fname=str(japanese_font)).get_name()
    rcParams["font.family"] = [family_name, "DejaVu Sans"]
    rcParams["axes.unicode_minus"] = False
    print("図の日本語フォント / figure font:", family_name)
    print("  ->", japanese_font)

In [ ]:
import cv2
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

def show(images, titles=None, figsize=(14, 5)):
    """画像を並べて表示する / display images side by side."""
    if not isinstance(images, (list, tuple)):
        images = [images]
    if titles is None:
        titles = [""] * len(images)
    fig, axes = plt.subplots(1, len(images), figsize=figsize)
    if len(images) == 1:
        axes = [axes]
    for ax, image, title in zip(axes, images, titles):
        ax.imshow(image)
        ax.set_title(title, fontsize=10)
        ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout()
    plt.show()


def detections_to_frame(result):
    """検出結果を pandas の表にする / convert detections into a DataFrame.

    ``result.boxes.data`` の 6 列は
    (x1, y1, x2, y2, confidence, class_id) です。
    """
    columns = ["class_id", "name", "confidence", "x1", "y1", "x2", "y2"]
    boxes = result.boxes
    if len(boxes) == 0:
        return pd.DataFrame(columns=columns)

    data = boxes.data.cpu().numpy()
    return pd.DataFrame({
        "class_id": data[:, 5].astype(int),
        "name": [result.names[int(class_id)] for class_id in data[:, 5]],
        "confidence": data[:, 4].round(4),
        "x1": data[:, 0].round(2),
        "y1": data[:, 1].round(2),
        "x2": data[:, 2].round(2),
        "y2": data[:, 3].round(2),
    })

In [ ]:
import shutil
from pathlib import Path

MODELS_DIR = ASSETS / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

def load_pretrained_yolo(model_name="yolo26n.pt"):
    """学習済みモデルを読み込む。初回だけダウンロードする。
    Load a pretrained model, downloading it only on the first run."""
    from ultralytics import YOLO

    weights_path = MODELS_DIR / model_name
    if not weights_path.is_file():
        print(f"初回のみダウンロードします / downloading {model_name} (first run only)...")
        print("  ネットワークに接続できない場合は、次を確認してください:")
        print("  - 社内プロキシの設定 / corporate proxy settings")
        print("  - もう一度このセルを実行する / run this cell again")
        model = YOLO(model_name)          # ダウンロードはカレントフォルダに行われる
        downloaded = Path(model_name)
        if downloaded.is_file():
            shutil.move(str(downloaded), str(weights_path))
            print("保存しました / cached to:", weights_path)
        else:
            raise RuntimeError(
                f"モデルを取得できませんでした / could not obtain {model_name}. "
                "ネットワークを確認して再実行してください / check the network and retry."
            )
    return YOLO(str(weights_path))

In [ ]:
import ultralytics
model = load_pretrained_yolo("yolo26n.pt")
sample_image = Path(ultralytics.__file__).parent / "assets" / "bus.jpg"
print("準備完了 / ready. image:", sample_image.is_file())

---

## Part 1　confidence のしきい値 / The confidence threshold

#### 日本語：しきい値は「どこまでを検出とみなすか」
**日本語：** モデルは、たくさんの候補を出します。その中から **confidence が
しきい値以上のものだけを採用** します。

| しきい値 | 結果 |
|---|---|
| 0.90 | ごく確かな検出だけ。**見逃し（偽陰性）が増える** |
| 0.25 | 標準的。バランスが良い |
| 0.05 | ほぼ全部採用。**誤検出（偽陽性）が増える** |

**日本語：** しきい値は **モデルではなく人間が決める** 数値です。
「見逃したくない」のか「誤報を減らしたい」のかで、変えるべき値が変わります。

**English:** The model produces many candidates; only those with **confidence above the threshold**
are kept.

| Threshold | Effect |
|---|---|
| 0.90 | only very certain detections; **more misses (false negatives)** |
| 0.25 | a common default with a reasonable balance |
| 0.05 | almost everything kept; **more false alarms (false positives)** |

The threshold is **chosen by a human, not by the model**. Whether you fear missing a defect or fear
false alarms determines the right value.
#### English: A threshold decides what counts as a detection

In [ ]:
thresholds = [0.05, 0.10, 0.25, 0.40, 0.60, 0.80, 0.90]
rows = []

for threshold in thresholds:
    prediction = model.predict(source=str(sample_image), conf=threshold, verbose=False)[0]
    boxes = prediction.boxes
    rows.append({
        "threshold": threshold,
        "detections": len(boxes),
        "classes": ", ".join(sorted({prediction.names[int(c)] for c in boxes.cls.tolist()})) if len(boxes) else "-",
        "max_conf": round(float(boxes.conf.max().item()), 3) if len(boxes) else 0.0,
    })

import pandas as pd
threshold_table = pd.DataFrame(rows)
print(threshold_table.to_string(index=False))

### 実行結果の例 / Expected output

**日本語：** 次のような結果になります（数値は環境によって多少変わります）。

```text
threshold  detections                                     classes  max_conf
      0.05           9 bus, handbag, person, skateboard, stop sign     0.881
      0.10           6                      bus, person, stop sign     0.881
      0.25           5                                 bus, person     0.881
      0.40           5                                 bus, person     0.881
      0.60           5                                 bus, person     0.881
      0.80           4                                 bus, person     0.881
      0.90           0                                           -     0.000
```

**English:** You should see something like the following (exact numbers vary slightly between machines).

```text
threshold  detections  classes                                     max_conf
     0.05           9  bus, handbag, person, skateboard, stop sign          0.881
     0.10           6  bus, person, stop sign                               0.881
     0.25           5  bus, person                                        0.881
     0.80           4  bus, person                                        0.881
     0.90           0  -                                                  0.000
```

#### 日本語：気づいてほしいこと
**日本語：** しきい値を 0.90 にすると、**バスの検出（confidence 0.881）も消えます**。
0.881 < 0.90 だからです。

**「しきい値を上げる＝厳しくする」は、正しいものまで捨てる** ことでもあります。

**English:** At a threshold of 0.90 the **bus itself disappears**, because its confidence (0.881)
is below 0.90.

Raising a threshold is not simply "being stricter": it also discards correct detections.
#### English: What you should notice

---

## Part 2　座標を読む / Reading the coordinates

#### 日本語：座標から何が分かるか
**日本語：** 座標を読むと、次のことが分かります。

- 欠陥の **位置**（画像のどこか）
- 欠陥の **大きさ**（幅・高さ）
- 画像の端に接しているか（**途中で切れている** 可能性）
- 同じ位置に複数の検出があるか（**重複検出**）

**English:** Coordinates tell you:

- **where** the object is
- **how large** it is (width and height)
- whether it touches the border (possibly **cut off**)
- whether several detections sit at the same place (**duplicates**)
#### English: What coordinates tell you

In [ ]:
prediction = model.predict(source=str(sample_image), conf=0.25, verbose=False)[0]
image_height, image_width = prediction.orig_shape

print("画像サイズ / image size (height, width):", prediction.orig_shape)
print()
print(f"{'class':<8} {'w':>7} {'h':>7} {'area':>9} {'w%':>6} {'h%':>6} {'touch border':>14}")
print("-" * 66)
for box in prediction.boxes:
    x1, y1, x2, y2 = [float(v) for v in box.xyxy[0].tolist()]
    name = prediction.names[int(box.cls.item())]
    width = x2 - x1
    height = y2 - y1
    touches = (x1 <= 1) or (y1 <= 1) or (x2 >= image_width - 1) or (y2 >= image_height - 1)
    print(f"{name:<8} {width:>7.1f} {height:>7.1f} {width * height:>9.0f} "
          f"{width / image_width * 100:>5.1f}% {height / image_height * 100:>5.1f}% "
          f"{'YES' if touches else 'no':>14}")

#### 日本語：大きさを比で表す
**日本語：** 実寸（ピクセル）で比べると、画像サイズが変わったときに意味が分かりにくくなります。
**画像に対する割合（%）** で見ると、画像サイズが変わっても比較できます。

**English:** Absolute pixels are hard to compare when the image size changes. Expressing the size as
a **percentage of the image** makes comparisons stable.
#### English: Express size as a ratio

---

## Part 3　画像を変えて比べる / Comparing different images

In [ ]:
second_image = Path(ultralytics.__file__).parent / "assets" / "zidane.jpg"

comparison = []
for label, path in [("bus.jpg", sample_image), ("zidane.jpg", second_image)]:
    if not path.is_file():
        print("画像がありません / missing:", path)
        continue
    r = model.predict(source=str(path), conf=0.25, verbose=False)[0]
    names = [r.names[int(c)] for c in r.boxes.cls.tolist()]
    comparison.append({
        "image": label,
        "detections": len(names),
        "unique_classes": ", ".join(sorted(set(names))) if names else "-",
        "mean_conf": round(float(r.boxes.conf.mean().item()), 3) if names else 0.0,
    })
    show([cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB)], [f"{label}: {len(names)} detections"], figsize=(7, 5))

print(pd.DataFrame(comparison).to_string(index=False))

---

## Part 4　偽陽性と偽陰性 / False positives and false negatives

#### 日本語：4 つの場合
**日本語：** 検出結果は、必ず次の 4 つのどれかになります。

| | 実際に欠陥がある | 実際は正常 |
|---|---|---|
| **検出した** | 正しい検出（true positive） | **偽陽性**（false positive, 誤検出） |
| **検出しない** | **偽陰性**（false negative, 見逃し） | 正しい（true negative） |

**製造検査では、偽陰性（見逃し）が特に深刻** です。不良品を出荷してしまうからです。
一方、偽陽性が多いと、ラインが止まりすぎて現場が疲弊します。

**日本語：** ですから **どちらをどれだけ許容するか** を、事前に決めておく必要があります。

**English:** Every detection falls into one of four cases.

| | defect present | actually good |
|---|---|---|
| **detected** | true positive | **false positive** |
| **not detected** | **false negative** | true negative |

In manufacturing, **false negatives are the serious ones** — a defective product ships.
Too many false positives, however, stop the line constantly and exhaust the operators.

You must therefore decide **in advance how much of each you can tolerate**.
#### English: The four cases

#### 日本語：confidence と正しさは別物
**日本語：** 次の 2 つは **どちらも起こります**。

- **低い confidence の検出が正しい** — 小さくて見えにくいが、実在するもの
- **高い confidence の検出が間違っている** — モデルが見た目に騙されている

**日本語：** だから「confidence が高いから大丈夫」とは言えません。
**必ず画像を見て確認する** ことが、この研修でも実務でも基本です。

**English:** Both of these happen:

- **a low-confidence detection can be correct** — small and faint, but real
- **a high-confidence detection can be wrong** — the model is fooled by appearance

So "confidence is high, therefore it is fine" is not a valid argument.
**Always check the image.**
#### English: Confidence and correctness are different

#### 日本語：失敗例を記録する
**日本語：** 下のセルは、`bus.jpg` の結果から **confidence の低い検出** を
一覧にします。低いものほど「疑わしい」候補です。

**日本語：** 演習では、この中から 1 件選んで、
**なぜ低いのか**（小さすぎる、隠れている、見た目が似ている）を考えます。

**English:** The next cell lists the **lowest-confidence detections** from `bus.jpg`.
Lower values are the most questionable candidates.

In the exercise you will pick one and explain **why** it is low: too small, occluded, or visually
similar to the class.
#### English: Record a failure case

In [ ]:
low_conf = model.predict(source=str(sample_image), conf=0.05, verbose=False)[0]
rows = []
for box in low_conf.boxes:
    x1, y1, x2, y2 = [float(v) for v in box.xyxy[0].tolist()]
    rows.append({
        "class": low_conf.names[int(box.cls.item())],
        "confidence": round(float(box.conf.item()), 3),
        "width_px": round(x2 - x1, 1),
        "height_px": round(y2 - y1, 1),
    })

low_table = pd.DataFrame(rows).sort_values("confidence").reset_index(drop=True)
print(low_table.to_string(index=False))
print()
print("最小の confidence / lowest confidence:", low_table.iloc[0]["confidence"] if len(low_table) else "-")

---

## Part 5　解像度と小さな物体 / Resolution and small objects

#### 日本語：画像を小さくすると検出できなくなる
**日本語：** モデルは入力を一定の大きさに変換して処理します。
元の画像が小さいと、**小さな物体は数ピクセルしか残りません**。

第 1 日で見たとおり、物体が数ピクセルを下回ると、もう区別できません。

**日本語：** 次のセルは、画像を意図的に縮小してから推論し、検出数がどう変わるかを見ます。

**English:** Models resize their input to a fixed size. If the source image is small, **small
objects survive as only a few pixels**, and as you saw on Day 1, once an object is smaller than a
few pixels it cannot be distinguished.

The next cell downsizes the image before inference and measures what happens to the detection count.
#### English: Shrinking an image loses detections

In [ ]:
source = cv2.imread(str(sample_image))
base_height, base_width = source.shape[:2]

scale_rows = []
work_dir = COURSE_ROOT / "submissions" / "day03"
work_dir.mkdir(parents=True, exist_ok=True)

for scale in [1.0, 0.5, 0.25, 0.15, 0.10]:
    new_size = (int(base_width * scale), int(base_height * scale))
    resized = cv2.resize(source, new_size, interpolation=cv2.INTER_AREA)
    temp_path = work_dir / f"scaled_{int(scale * 100)}.png"
    cv2.imwrite(str(temp_path), resized)

    r = model.predict(source=str(temp_path), conf=0.25, verbose=False)[0]
    scale_rows.append({
        "scale": scale,
        "image_width": new_size[0],
        "image_height": new_size[1],
        "detections": len(r.boxes),
        "classes": ", ".join(sorted({r.names[int(c)] for c in r.boxes.cls.tolist()})) or "-",
    })

print(pd.DataFrame(scale_rows).to_string(index=False))

#### 日本語：なぜ 0 件になるのか
**日本語：** 縮小すると、人の横幅が数十ピクセルから数ピクセルになります。
モデルは「人の形」を認識できなくなります。

**日本語：** これは PVC の 0.3 mm 欠陥で、**さらに深刻** になります。
欠陥が 1 画素未満なら、AI の性能以前の問題です。

**English:** After downscaling, a person only a few pixels wide no longer looks like a person to
the model.

For a 0.3 mm PVC defect this is **far worse**: if the defect is smaller than one pixel, no AI can
help. That is an optics problem, not a model problem.
#### English: Why the count drops to zero

---

## 演習 / Exercises

#### 日本語：演習 1（基本）：しきい値の表を作る
**日本語：** Part 1 の表を、`zidane.jpg` でも作り直してください。
しきい値 0.05〜0.90 で、検出数の変化を記録してください。

**English:** Recreate the Part 1 table for `zidane.jpg` over thresholds 0.05-0.90 and record how the
detection count changes.
#### English: Exercise 1 (basic): build the threshold table
Done when the table has 7 rows and you note at which threshold detections drop to zero.

In [ ]:
# 演習 1 / Exercise 1
rows = []
for threshold in [0.05, 0.10, 0.25, 0.40, 0.60, 0.80, 0.90]:
    # r = model.predict(...)
    rows.append({"threshold": threshold, "detections": 0})
print(pd.DataFrame(rows).to_string(index=False))

#### 日本語：演習 2（中級）：失敗例を 1 件記録する
**日本語：** 上の「低い confidence の一覧」から 1 件選び、記録してください。

| 項目 | 記入 |
|---|---|
| クラス | |
| confidence | |
| 大きさ（幅×高さ） | |
| なぜ低いと思うか | |
| 本当に物体か（画像を見た判断） | |

**English:** Pick one entry from the low-confidence table and fill in the record above.
#### English: Exercise 2 (intermediate): record one failure case
Done when all five fields are filled using your own observation of the image.

In [ ]:
# 演習 2 / Exercise 2
record = {
    "class": "",
    "confidence": 0.0,
    "size_px": "",
    "why_low": "",
    "is_real_object": "",
}
for key, value in record.items():
    print(f"{key:<16} {value if value not in ('', 0.0) else '(未記入 / not written)'}")

#### 日本語：演習 3（発展）：PVC 検査に必要な解像度を提案する
**日本語：** 幅 1,300 mm のシートで 0.3 mm の欠陥を、**最低 3 画素** で捉えたいとします。

1. 横方向に必要な画素数を計算してください。
2. その画像を YOLO にそのまま入力できますか。できない場合、どう工夫しますか。
3. Part 5 の結果を根拠に、「画像を縮小して推論する」ことの問題点を説明してください。

**English:** To capture a 0.3 mm defect on a 1,300 mm sheet with **at least 3 pixels**:

1. compute the required horizontal pixel count
2. can that image be fed to YOLO directly? If not, what would you do?
3. using the Part 5 results as evidence, explain the problem with "shrink the image before inference"
#### English: Exercise 3 (challenge): propose the resolution needed for PVC
Done when the pixel count is stated and the shrinking problem is explained with evidence.

In [ ]:
# 演習 3 / Exercise 3
sheet_width_mm = 1300.0
defect_mm = 0.3
min_pixels = 3

required_px = 0.0     # <- 計算 / compute
print("1. 必要な横方向の画素数 / required pixels:", required_px)
print("2. そのまま入力できるか / usable directly? :", "")
print("3. 縮小の問題点 / problem with shrinking :", "")

---

#### 日本語：理解度チェック / Understanding questions
**日本語：**

1. しきい値を 0.9 にすると、なぜ正しい検出まで消えるのですか。
2. 偽陽性と偽陰性を、製造現場の言葉で説明してください。
3. 「confidence が高いから正しい」と言えないのはなぜですか。
4. 画像を縮小すると検出数が減る理由を、2 つ挙げてください。
5. PVC 検査で「見逃し」と「誤報」のどちらを優先して減らすべきですか。あなたの意見と理由を書いてください。

**English:**

1. Why do correct detections disappear at a threshold of 0.9?
2. Explain false positives and false negatives in shop-floor language.
3. Why can you not say "high confidence means correct"?
4. Give two reasons why shrinking an image reduces detections.
5. In PVC inspection, which should you reduce first: misses or false alarms? Give your view and reasons.
#### English: Understanding questions
Model answers: `docs/instructor/day03_instructor_guide.md`.

---

## 困ったときは / Troubleshooting

**日本語：** エラーが出たら、まず「どのセルで」「どの行で」「どんなメッセージが出たか」を 書き留めてください。提出物にも記録します。

**English:** When an error appears, first write down *which cell*, *which line*, and *what the message says*. You will also record it in your submission.

| 症状（日本語） | Symptom (English) | 対処（日本語） | What to do (English) |
| --- | --- | --- | --- |
| 表が空になる | The table is empty | `conf` が高すぎる可能性があります。0.05 まで下げて確認してください。 | `conf` may be too high; lower it to 0.05. |
| `model` が未定義 | `model` is not defined | Part 0 のモデル読み込みセルを実行してください。 | Run the model-loading cell in Part 0. |
| 縮小した画像でエラーが出る | An error occurs with the resized images | `int()` で整数にしていますが、極端に小さいと 0 になります。`scale` を 0.10 未満にしないでください。 | Very small sizes can become 0; do not go below `scale=0.10`. |
| CSV を Excel で開くと文字化けする | The CSV is garbled in Excel | `encoding="utf-8-sig"` を付けて保存してください。 | Save with `encoding="utf-8-sig"`. |
| 毎回ダウンロードが始まる | It downloads every time | `assets/models/` にファイルがあるか確認してください。無ければ手動で置いてください。 | Check that `assets/models/` contains the file; if not, place it there manually. |

## 完了チェックリスト / Completion checklist

**日本語：** すべてにチェックが付いたら、このノートブックは完了です。

**English:** This notebook is complete when every box is checked.

- [ ] Part 1〜Part 5 を読み、すべてのコードセルを実行した
      - [ ] Read Parts 1-5 and ran every code cell
- [ ] しきい値ごとの検出数の表を作った
      - [ ] Built the threshold-versus-detections table
- [ ] 座標から幅・高さ・割合を求めた
      - [ ] Computed width, height and ratio from coordinates
- [ ] 解像度を下げたときの検出数の変化を確認した
      - [ ] Observed the effect of downscaling
- [ ] 演習 1〜3 を実施した
      - [ ] Completed exercises 1-3
- [ ] 失敗例を 1 件以上記録した
      - [ ] Recorded at least one failure case
- [ ] 理解度チェックの 5 問に答えた
      - [ ] Answered the 5 questions
- [ ] ノートブックを保存した
      - [ ] Saved the notebook

## さらに学ぶ / Further learning（任意 / optional）

### 参考リンク / Reference links

- [Ultralytics Predict モード](https://docs.ultralytics.com/modes/predict/)
- [Ultralytics 検証モード](https://docs.ultralytics.com/modes/val/)
- [COCO 評価指標の考え方](https://cocodataset.org/#detection-eval)

### 追加チャレンジ / Extra challenge（必須ではありません / not mandatory）

**日本語：** `iou` のしきい値（`model.predict(..., iou=...)`）を変えると、重複検出がどう変わるか調べてください。

**English:** Change the NMS `iou` threshold in `model.predict(..., iou=...)` and observe how duplicate detections change.